# Posterior for local recombination rate using SBI (real genes)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from Bio import Phylo
import torch
from torch.distributions import Uniform
from sbi.utils.user_input_checks import MultipleIndependent
from sbi.inference import NPE_C
from sbi.analysis import plot_summary
import sys
sys.path.append('../pysimARG')
from clonal_genealogy import ClonalTree
from newick_to_tree import newick_to_tree
from discrete_uniform import DiscreteUniform

torch_device = "cpu"

c:\Users\u2008181\likelihood-free\sbi_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load simulation data

Load staph gene data and clonal tree.

In [2]:
np.random.seed(100)
clonal_tree = ClonalTree(n=110)

# Load phylo tree and convert to ClonalTree format
phylo_tree = Phylo.read("../data/staph/saureus_clonal.nwk", "newick")
Phylo.draw_ascii(phylo_tree)

edge, node_height = newick_to_tree(phylo_tree)
clonal_tree.edge = edge
clonal_tree.node_height = node_height
clonal_tree.height = np.max(node_height)
clonal_tree.length = np.sum(edge[:, 2])

                                                                   , C1139
                                                                  ,|
                                                                  |, C9658
                                                                  ||
                                                                  |, C1282
                                                                  ||
                      ____________________________________________|, C1280
                     |                                            ||
                     |                                            || C1080
                     |                                            |
                     |                                            |_ C1079
                     |                                            |
                     |                                            , C1147
                     |                                          

In [3]:
x_obs_df = pd.read_csv("../data/staph/core_gene_summary_stats.csv", index_col=0)
x_obs_np = x_obs_df.to_numpy()
x_obs_torch = torch.tensor(x_obs_np, device=torch_device)
x_obs_torch = x_obs_torch.to(torch.float32)
x_obs_torch.shape, x_obs_torch.dtype

(torch.Size([1983, 46]), torch.float32)

## Load simulation data

In [4]:
theta1 = np.loadtxt('../data/staph/ClonalOrigin_sim/theta1.csv', delimiter=",")
x1 = np.loadtxt('../data/staph/ClonalOrigin_sim/x1.csv', delimiter=",")

theta2 = np.loadtxt('../data/staph/ClonalOrigin_sim/theta2.csv', delimiter=",")
x2 = np.loadtxt('../data/staph/ClonalOrigin_sim/x2.csv', delimiter=",")

theta3 = np.loadtxt('../data/staph/ClonalOrigin_sim/theta3.csv', delimiter=",")
x3 = np.loadtxt('../data/staph/ClonalOrigin_sim/x3.csv', delimiter=",")

x = np.vstack([x1, x2, x3])
theta = np.vstack([theta1, theta2, theta3])

print(theta.shape, x.shape)

(50000, 2) (50000, 46)


In [5]:
theta = torch.tensor(theta, device=torch_device)
theta = theta.to(torch.float32)
theta_numpy = theta.cpu().numpy()

x = torch.tensor(x, device=torch_device)
x = x.to(torch.float32)
x_numpy = x.cpu().numpy()

## NPE

### Create prior to pass range knowledge to NPE

In [7]:
prior_rho = Uniform(low=torch.tensor([0.0]), high=torch.tensor([0.1]))
prior_L = DiscreteUniform(low=torch.tensor([20.0]), high=torch.tensor([10000.0]))

prior = MultipleIndependent(
    dists=[prior_rho, prior_L],
    validate_args=False,
    device=torch_device
)

In [8]:
seed = 100
num_posterior_samples=1000
learning_rate = 0.0005

inference = NPE_C(prior=prior, density_estimator="nsf", device=torch_device)
torch.manual_seed(seed)
np.random.seed(seed)

In [ ]:
density_estimator = inference.append_simulations(theta, x).train(
    max_num_epochs=500, learning_rate=learning_rate
)
posterior = inference.build_posterior(density_estimator)

In [ ]:
fig, axes = plot_summary(
    inference, 
    tags=["training_loss", "validation_loss"], 
    figsize=(8, 4)
)
plt.title("Training and Validation Loss")
plt.show()

### Plug in observation data to get posterior

In [9]:
theta_post = np.full((x_obs_torch.shape[0], num_posterior_samples, 2), np.nan)
theta_post.shape

(1983, 1000, 2)

In [11]:
ignore_indices = []
no_segregation = []
out_stats = dict()
for i in range(x_obs_torch.shape[0]):
    ignore_i = False
    out_index = []
    for j in range(46):
        max_j = torch.max(x[:, j])
        min_j = torch.min(x[:, j])
        obs_j = x_obs_torch[i, j]
        if obs_j < min_j or obs_j > max_j:
            ignore_i = True
            out_index.append(j)
        if j == 33 and obs_j == 0:
            no_segregation.append(i)
    if ignore_i or torch.isnan(x_obs_torch[i, :]).any():
        print(f"Observation {i} is outside the range of simulated data.")
        ignore_indices.append(i)
        out_stats[i] = out_index

Observation 5 is outside the range of simulated data.
Observation 6 is outside the range of simulated data.
Observation 7 is outside the range of simulated data.
Observation 8 is outside the range of simulated data.
Observation 20 is outside the range of simulated data.
Observation 22 is outside the range of simulated data.
Observation 27 is outside the range of simulated data.
Observation 29 is outside the range of simulated data.
Observation 30 is outside the range of simulated data.
Observation 31 is outside the range of simulated data.
Observation 35 is outside the range of simulated data.
Observation 39 is outside the range of simulated data.
Observation 47 is outside the range of simulated data.
Observation 64 is outside the range of simulated data.
Observation 82 is outside the range of simulated data.
Observation 85 is outside the range of simulated data.
Observation 86 is outside the range of simulated data.
Observation 87 is outside the range of simulated data.
Observation 89

In [12]:
len(ignore_indices), len(no_segregation), len(out_stats)

(300, 18, 300)

In [23]:
out_stats[idx]

[]

In [ ]:
np.where(np.isnan(x_obs_np).any(axis=0))[0]

array([37, 41])

In [32]:
np.where(np.isnan(np.delete(x_obs_np, no_segregation, axis=0)).any(axis=0))[0]

array([41])

In [37]:
out_index_all = []
for i in range(len(ignore_indices)):
    idx = ignore_indices[i]
    out_index_all += out_stats[idx]
    print(out_stats[idx])

[16, 17, 20, 21]
[20]
[20, 21]
[20, 21]
[16, 17, 20, 21]
[16, 17, 19, 22]
[16, 17, 22]
[16, 17, 20]
[20]
[16, 20]
[16, 17, 20, 21]
[17]
[16, 17, 20, 21]
[20]
[0, 1, 2, 3, 4, 5, 16, 17, 18, 19, 20, 21, 34, 35, 36]
[16, 17, 20, 21]
[16, 17, 20]
[20]
[16, 17, 20]
[20, 21]
[4, 5, 16, 18, 19, 20, 21]
[16]
[20]
[20]
[4, 5, 18, 20, 21]
[20, 21]
[20, 21]
[]
[20, 21]
[]
[]
[16, 17, 20, 21]
[16, 20, 21]
[16, 17, 20, 21]
[16, 20, 21]
[16, 17, 20, 21]
[16, 17, 20, 21]
[20]
[16, 17, 20, 21]
[16, 20, 21]
[16, 17, 20, 21]
[16, 17, 20, 21]
[16, 17, 20, 21]
[16, 17, 20, 21]
[16, 17, 20, 21]
[20, 21]
[20]
[20]
[20, 21]
[17, 20]
[16]
[38]
[16, 17]
[20]
[20, 21]
[]
[20]
[16, 17, 20]
[16, 17, 20, 21]
[20, 21]
[16, 17, 20, 21]
[20, 21]
[20, 21]
[20]
[16, 17, 20]
[20]
[20, 21]
[20, 21]
[20]
[]
[1, 16, 17, 20, 21, 36]
[16, 17, 20]
[16, 17, 20, 21]
[0, 1, 16, 17, 20, 21, 36]
[16]
[20, 21]
[16, 20]
[17, 21]
[16, 17, 20, 21]
[16, 20, 21]
[16, 17, 20]
[16, 17]
[17, 21]
[16, 17, 20, 21]
[16, 17, 20, 21]
[1, 5, 16,

In [39]:
from collections import Counter

integer_counts = Counter(out_index_all)
print(integer_counts)

Counter({20: 227, 21: 161, 16: 155, 17: 121, 36: 16, 1: 8, 5: 8, 38: 8, 0: 6, 4: 6, 18: 4, 35: 4, 19: 3, 22: 2, 2: 1, 3: 1, 34: 1, 33: 1, 42: 1, 44: 1, 23: 1})


In [62]:
ignore_indices[14]

82

In [65]:
x_obs_np[82, :]

array([ 8.13540744e+02,  2.79148017e+02,  6.43670491e+04,  2.25966104e+04,
        4.47487413e+03,  1.53779071e+03,  2.23000000e+03,  8.06000000e+02,
        1.24314774e-02,  1.21537799e-02,  9.83573990e-01,  9.83830130e-01,
        6.83792386e-02,  6.69536184e-02,  3.40759757e-02,  3.50923023e-02,
        1.11019008e+01,  1.02901653e+01,  7.41214025e+02,  7.73893819e+02,
        7.03886255e+01,  5.81589719e+01,  9.00000000e+00,  2.20000000e+01,
        1.49219097e-02,  1.31419735e-02,  9.96255410e-01,  9.88370140e-01,
        9.46083676e-02,  7.42771033e-02,  1.20967742e-02,  2.80970626e-02,
        4.20907840e-01,  5.86677815e-02,  7.98385223e+01,  5.92386989e+01,
        1.11307313e+03, -8.66590832e-01,  2.85714286e-02,  5.22565321e-02,
        4.60000000e+01,  6.80088773e-02,  2.81708842e-02,  1.87546735e-02,
        6.73229977e-02,  7.17600000e+03])

In [59]:
torch.max(x[:, 36]), torch.min(x[:, 36])

(tensor(412.5512), tensor(0.))

In [56]:
torch.max(x[:, 34]), torch.min(x[:, 34])

(tensor(69.7876), tensor(0.))

In [41]:
sorted(integer_counts.keys())

[0, 1, 2, 3, 4, 5, 16, 17, 18, 19, 20, 21, 22, 23, 33, 34, 35, 36, 38, 42, 44]

In [ ]:
skip_indices = []
for i in range(x_obs_torch.shape[0]):
    if i in ignore_indices:
        skip_indices.append(i)
        continue
    try:
        theta_post_torch = posterior.sample((num_posterior_samples,), x=x_obs_torch[i, :],
                                            show_progress_bars=False, max_sampling_time=5.0)
        theta_post[i, :, :] = theta_post_torch.cpu().numpy()
    except RuntimeError as e:
        print(f"Skipping observation {i} due to sampling timeout: {e}")
        skip_indices.append(i)
        continue

In [ ]:
plt.figure(figsize=(12, 8))
sns.set_style('whitegrid')
for i in range(0, x_obs_torch.shape[0]):
    if i in skip_indices:
        continue
    sns.kdeplot(theta_post[i, :, 0], color='blue', linewidth=0.3, alpha=0.5)
plt.axvline(x=0.0003887292720131564, color='red', linestyle='dashed', label='True value')
plt.xlabel(r'$\rho_s$')
plt.xlim(0.0, 0.01)
plt.legend()
plt.show()

In [ ]:
theta_post2 = np.full((x_obs_torch.shape[0], num_posterior_samples, 2), np.nan)
theta_post2.shape

In [ ]:
for i in range(x_obs_torch.shape[0]):
    if i in ignore_indices:
        continue

    theta_post2_torch = posterior.sample((num_posterior_samples,), x=x_obs_torch[i, :],
                                        show_progress_bars=False, reject_outside_prior=False)
    theta_post2[i, :, :] = theta_post2_torch.cpu().detach().numpy()

In [ ]:
plt.figure(figsize=(12, 8))
sns.set_style('whitegrid')
for i in range(0, x_obs_torch.shape[0]):
    if i in ignore_indices:
        continue
    sns.kdeplot(theta_post2[i, :, 0], color='blue', linewidth=0.3, alpha=0.5)
plt.axvline(x=0.0003887292720131564, color='red', linestyle='dashed', label='True value')
plt.xlabel(r'$\rho_s$')
plt.xlim(0.0, 0.01)
plt.legend()
plt.show()

### Plot bar plots by position

In [ ]:
gene_info = pd.read_csv("../data/staph/core_gene_summary.csv", index_col=0)
gene_info_np = gene_info.to_numpy()
gene_info.head()

In [ ]:
plot_xlim = [0.0, 2902619.0]
samples_x1 = np.delete(gene_info_np[:, 1], skip_indices)
samples_x2 = np.delete(gene_info_np[:, 1], ignore_indices)

theta_post_nonan = np.delete(theta_post[:, :, 0], skip_indices, axis=0)
theta_post2_nonan = np.delete(theta_post2[:, :, 0], ignore_indices, axis=0)

In [ ]:
theta_post_nonan.shape, theta_post2_nonan.shape, samples_x1.shape, samples_x2.shape

In [ ]:
posterior1_median = np.median(theta_post_nonan, axis=1)
posterior2_median = np.median(theta_post2_nonan, axis=1)
posterior1_median.shape, posterior2_median.shape

In [ ]:
ci_lower_bounds1 = np.percentile(theta_post_nonan, 2.5, axis=1)
ci_upper_bounds1 = np.percentile(theta_post_nonan, 97.5, axis=1)
ci_lower_bounds2 = np.percentile(theta_post2_nonan, 2.5, axis=1)
ci_upper_bounds2 = np.percentile(theta_post2_nonan, 97.5, axis=1)
ci_lower_bounds1.shape, ci_upper_bounds1.shape, ci_lower_bounds2.shape, ci_upper_bounds2.shape

In [ ]:
lower1_errors = posterior1_median - ci_lower_bounds1
upper1_errors = ci_upper_bounds1 - posterior1_median
lower2_errors = posterior2_median - ci_lower_bounds2
upper2_errors = ci_upper_bounds2 - posterior2_median
yerr1 = [lower1_errors, upper1_errors]
yerr2 = [lower2_errors, upper2_errors]

In [ ]:
plt.figure(figsize=(12, 6))
plt.errorbar(samples_x1, posterior1_median, yerr=yerr1, fmt='o', color='black', 
             ecolor='gray', capsize=4, elinewidth=1.5, markersize=6, 
             label='Posterior Median ± 95% CI')

plt.title("Posterior Medians with 95% Confidence Intervals", fontsize=14)
plt.xlabel("X values", fontsize=12)
plt.ylabel("Posterior Median", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.axhline(y=0.0003887292720131564, color='red', linestyle='dashed', label='True value')
plt.xlim(plot_xlim)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(18, 6))
plt.errorbar(samples_x2, posterior2_median, yerr=yerr2, fmt='o', color='black', 
             ecolor='gray', capsize=4, elinewidth=1.5, markersize=6, 
             label='Posterior Median ± 95% CI')

plt.title("Posterior Medians with 95% Confidence Intervals for local recombination rate", fontsize=14)
plt.xlabel("Position (bp)", fontsize=12)
plt.ylabel(r'$\rho_s$', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.axhline(y=0.0003887292720131564, color='red', linestyle='dashed', label='Global value from ClonalFrameML')
plt.xlim(plot_xlim)
plt.legend(loc='upper left')
plt.show()

In [ ]:
len(ignore_indices)

In [ ]:
samples_l2 = np.delete(gene_info_np[:, 0], ignore_indices)
theta_post_l2_nonan = np.delete(theta_post2[:, :, 1], ignore_indices, axis=0)
samples_l2.shape, theta_post_l2_nonan.shape

In [ ]:
posterior_l2_median = np.median(theta_post_l2_nonan, axis=1)
posterior_l2_median.shape

In [ ]:
ci_lower_bounds_l2 = np.percentile(theta_post_l2_nonan, 2.5, axis=1)
ci_upper_bounds_l2 = np.percentile(theta_post_l2_nonan, 97.5, axis=1)
ci_lower_bounds_l2.shape, ci_upper_bounds_l2.shape

In [ ]:
lower_l2_errors = posterior_l2_median - ci_lower_bounds_l2
upper_l2_errors = ci_upper_bounds_l2 - posterior_l2_median
yerr_l2 = [lower_l2_errors, upper_l2_errors]

In [ ]:
plt.figure(figsize=(8, 8))
plt.errorbar(samples_l2, posterior_l2_median, yerr=yerr_l2, fmt='o', color='black', 
             ecolor='gray', capsize=4, elinewidth=1.5, markersize=6, 
             label='Posterior Median ± 95% CI')

exact_x = np.linspace(np.min(samples_l2), np.max(samples_l2), 400)
exact_y = exact_x

plt.title("Posterior Medians with 95% Confidence Intervals for Gene Length", fontsize=14)
plt.xlabel("True length (bp)", fontsize=12)
plt.ylabel("Estimated length (bp)", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.plot(exact_x, exact_y, color='red', linestyle='dashed')
plt.legend(loc='upper left')
plt.show()